# Model Comparison Notebook

This notebook brings together the trained Logistic Regression and Random Forest
models and runs the remaining evaluation steps from the project proposal:

1. Add the project root to `sys.path`
2. Load the processed dataset and recreate the shared train/test split
3. Build and train both models
4. K-Fold Cross-Validation (5 folds, ROC-AUC)
5. Threshold Analysis (precision/recall trade-off at multiple cut-offs)
6. Side-by-side Model Comparison
7. Final Output — visualisations and saved summary

In [ ]:
# Import Path so the notebook can locate the repository root.
from pathlib import Path
# Import sys so the notebook can modify Python's module search path.
import sys

# The notebook lives inside `notebooks/`, so the parent directory is the
# project root that contains the `src/` package.
PROJECT_ROOT = Path.cwd().resolve().parent
# Add the project root once so imports like `from src.models import ...`
# work inside this notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## Load the processed dataset and recreate the train/test split

The same `random_state=42` and `stratify=y` settings used in the individual
model notebooks are used here so the splits are identical and the comparison
is fair.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.config import PROCESSED_DATA_PATH
from src.preprocessing import prepare_features, build_preprocessor

# Read the processed churn dataset.
df = pd.read_csv(PROCESSED_DATA_PATH)

# Split the DataFrame into feature columns and the churn target.
X, y = prepare_features(df)

# Build the shared preprocessing transformer.
preprocessor = build_preprocessor()

# Recreate the same stratified train/test split used by both model notebooks.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Full dataset:  {X.shape[0]} rows, {X.shape[1]} features")
print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")

## Build and train both models

Both pipelines use the same preprocessor definition so the comparison reflects
only the model choice, not any difference in data preparation.

In [ ]:
from src.models import build_logistic_model, build_random_forest_model

# Build the logistic regression pipeline.
log_model = build_logistic_model(preprocessor)
log_model.fit(X_train, y_train)
print("Logistic Regression trained.")

# Build the random forest pipeline.
rf_model = build_random_forest_model(preprocessor)
rf_model.fit(X_train, y_train)
print("Random Forest trained.")

## K-Fold Cross-Validation

5-fold stratified cross-validation measures how consistently each model
performs across different data splits.  A low standard deviation means the
model generalises reliably and is not just performing well on one lucky split.

The full feature matrix `X` (not just `X_train`) is passed to `run_kfold_cv`
because `cross_val_score` creates its own folds internally.

In [ ]:
from src.evaluate import run_kfold_cv

# Run 5-fold cross-validation for Logistic Regression.
log_cv = run_kfold_cv(log_model, X, y, n_splits=5, scoring="roc_auc")

# Run 5-fold cross-validation for Random Forest.
rf_cv = run_kfold_cv(rf_model, X, y, n_splits=5, scoring="roc_auc")

print("Logistic Regression — 5-Fold CV ROC-AUC")
for i, score in enumerate(log_cv["scores"], 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"  Mean: {log_cv['mean']:.4f}  |  Std: {log_cv['std']:.4f}")

print()
print("Random Forest — 5-Fold CV ROC-AUC")
for i, score in enumerate(rf_cv["scores"], 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"  Mean: {rf_cv['mean']:.4f}  |  Std: {rf_cv['std']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Visualise the per-fold score distributions as a box plot so it is easy to
# see both the central tendency and the spread for each model.
fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [log_cv["scores"], rf_cv["scores"]],
    labels=["Logistic Regression", "Random Forest"],
    patch_artist=True,
    boxprops=dict(facecolor="lightsteelblue"),
    medianprops=dict(color="navy", linewidth=2),
)
ax.set_title("5-Fold Cross-Validation ROC-AUC", fontsize=13, fontweight="bold")
ax.set_ylabel("ROC-AUC Score")
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## Threshold Analysis

By default, sklearn predicts churn when the estimated probability is >= 0.5.
This cell evaluates precision, recall, and F1 at thresholds ranging from 0.1
to 0.7 to show how the trade-off changes.

A lower threshold catches more churners (higher recall) but also flags more
non-churners incorrectly (lower precision).  For a bank, missing an actual
churner is typically more costly than a false alarm, so a lower threshold may
be preferable in practice.

In [ ]:
from src.evaluate import run_threshold_analysis

# Run threshold analysis for both models on the held-out test set.
log_threshold = run_threshold_analysis(log_model, X_test, y_test)
rf_threshold  = run_threshold_analysis(rf_model,  X_test, y_test)

# Display results as a DataFrame for easier reading.
import pandas as pd

print("Logistic Regression — Threshold Analysis")
display(pd.DataFrame(log_threshold).round(4))

print("\nRandom Forest — Threshold Analysis")
display(pd.DataFrame(rf_threshold).round(4))

In [ ]:
# Plot precision and recall vs threshold for both models.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, results, title in zip(
    axes,
    [log_threshold, rf_threshold],
    ["Logistic Regression", "Random Forest"],
):
    thresholds  = [r["threshold"]  for r in results]
    precisions  = [r["precision"]  for r in results]
    recalls     = [r["recall"]     for r in results]
    f1_scores   = [r["f1"]         for r in results]

    ax.plot(thresholds, precisions, marker="o", label="Precision",  color="steelblue")
    ax.plot(thresholds, recalls,    marker="s", label="Recall",     color="darkorange")
    ax.plot(thresholds, f1_scores,  marker="^", label="F1-Score",   color="green")

    # Mark the default 0.5 threshold with a dashed vertical line.
    ax.axvline(x=0.5, color="grey", linestyle="--", linewidth=1, label="Default (0.5)")

    ax.set_title(f"{title}\nPrecision / Recall / F1 vs Threshold", fontsize=12, fontweight="bold")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Side-by-Side Model Comparison

This section collects all evaluation metrics in one place for a direct
comparison between the two models.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Generate predictions and probabilities for both models on the test set.
log_pred = log_model.predict(X_test)
rf_pred  = rf_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:, 1]
rf_prob  = rf_model.predict_proba(X_test)[:, 1]

# Build a comparison DataFrame so all metrics appear in one readable table.
comparison = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC (test)",
            "CV ROC-AUC Mean",
            "CV ROC-AUC Std",
        ],
        "Logistic Regression": [
            accuracy_score(y_test, log_pred),
            precision_score(y_test, log_pred),
            recall_score(y_test, log_pred),
            f1_score(y_test, log_pred),
            roc_auc_score(y_test, log_prob),
            log_cv["mean"],
            log_cv["std"],
        ],
        "Random Forest": [
            accuracy_score(y_test, rf_pred),
            precision_score(y_test, rf_pred),
            recall_score(y_test, rf_pred),
            f1_score(y_test, rf_pred),
            roc_auc_score(y_test, rf_prob),
            rf_cv["mean"],
            rf_cv["std"],
        ],
    }
).round(4)

display(comparison)

## Final Output — Visualisations

The cells below produce the final comparison charts:
- Confusion matrices for both models
- ROC curves on the same axes
- Metrics bar chart
- Random Forest feature importances

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Plot confusion matrices side by side.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, title in zip(
    axes,
    [log_pred, rf_pred],
    ["Logistic Regression", "Random Forest"],
):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Retained", "Churned"],
        yticklabels=["Retained", "Churned"],
    )
    ax.set_title(f"{title}\nConfusion Matrix", fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

# Plot ROC curves for both models on the same axes.
plt.figure(figsize=(8, 6))
for prob, label, color in [
    (log_prob, "Logistic Regression", "steelblue"),
    (rf_prob,  "Random Forest",       "darkorange"),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC = {auc:.4f})")

# Add the diagonal baseline that represents random guessing.
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves — Model Comparison", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Build a grouped bar chart so each metric can be compared at a glance.
metric_names = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
log_vals = [
    accuracy_score(y_test, log_pred),
    precision_score(y_test, log_pred),
    recall_score(y_test, log_pred),
    f1_score(y_test, log_pred),
    roc_auc_score(y_test, log_prob),
]
rf_vals = [
    accuracy_score(y_test, rf_pred),
    precision_score(y_test, rf_pred),
    recall_score(y_test, rf_pred),
    f1_score(y_test, rf_pred),
    roc_auc_score(y_test, rf_prob),
]

x = np.arange(len(metric_names))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w / 2, log_vals, w, label="Logistic Regression", color="steelblue")
b2 = ax.bar(x + w / 2, rf_vals,  w, label="Random Forest",       color="darkorange")

# Annotate each bar with its numeric value.
for bar in list(b1) + list(b2):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{bar.get_height():.3f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )

ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Extract feature names from the preprocessing pipeline.
# The ColumnTransformer stores the fitted transformers in the same order
# they were defined: categorical first, then numerical.
cat_features  = rf_model.named_steps["preprocessor"] \
                         .named_transformers_["cat"] \
                         .get_feature_names_out(
                             ["Geography", "Gender", "Card Type"]
                         ).tolist()
num_features  = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts",
                 "EstimatedSalary", "Satisfaction Score", "Point Earned"]
feature_names = cat_features + num_features

# Build a Series of feature importances sorted in descending order.
importances = pd.Series(
    rf_model.named_steps["classifier"].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

# Plot the top 12 most important features.
plt.figure(figsize=(10, 6))
importances.head(12).plot(kind="bar", color="steelblue", edgecolor="white")
plt.title("Random Forest — Top 12 Feature Importances", fontsize=13, fontweight="bold")
plt.ylabel("Importance Score")
plt.xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(importances.head(12).round(4))

## Save the comparison summary

The final comparison table and the cross-validation results are saved to
`results/model_comparison.txt` so they are available as a persistent artifact.

In [ ]:
from src.models import run_comparison_workflow
from src.config import PROCESSED_DATA_PATH

# Re-run the comparison workflow to produce the saved text summary.
# This calls the same logic shown above but also writes the results file.
comparison_results = run_comparison_workflow(data_path=PROCESSED_DATA_PATH)

print(f"Summary saved to: {comparison_results['comparison_path']}")